In [1]:
from __future__ import division
import pandas as pd
import numpy as np
from copy import deepcopy

import warnings
warnings.filterwarnings('ignore')

#!pip install py_stringmatching
#!pip install py_entitymatching
#!pip install py_stringsimjoin
import py_stringmatching as sm
import py_entitymatching as em
import py_stringsimjoin as ssj

from py_entitymatching.catalog import catalog_manager as cm

import random

# Introduzione

Questo Notebook riassume tutto il processo di Entity Resolution:
Blocking, Matching e Clustering.


Contiene i punti essenziali per discutere  l' Entity Resolution



# Funzioni utilizzate

In [2]:
######################CLUSTERING######################

import networkx as nx

def ClusterComponentiConnessi(MatchTable, TuttiInodi):

    MatchTable=deepcopy(MatchTable)
    MatchTable.columns=['A','B']

    Singleton = set(TuttiInodi) - set(MatchTable['A']).union(set(MatchTable['B']))

    # Creazione del grafo a partire dagli elementi della MatchTable
    G = nx.Graph()
    for _, row in MatchTable.iterrows():
        G.add_edge(row['A'], row['B'])
#        G.add_edge(row['A'], row['B'], weight=row['sim'])  # Aggiungi il peso (etichetta) basato su 'sim'

    # Aggiungi gli elementi singleton all'insieme dei nodi
    for element in Singleton:
        G.add_node(element)

    # Calcola i componenti connessi (clusters)
    clusters = list(nx.connected_components(G))

    # Creazione del DataFrame dei cluster
    cluster_data = {'ClusterKey': [], 'ClusterElement': []}
    for i, cluster in enumerate(clusters):
        for element in cluster:
            cluster_data['ClusterKey'].append(i + 1)
            cluster_data['ClusterElement'].append(element)

    cluster_df = pd.DataFrame(cluster_data)
    return cluster_df

In [3]:
def VisualizzaCluster2(Clusters):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI
### DELL'IDENTIFICATIVO DEL RECORD
### ADEGUARE IN BASE ALL'ESEMPIO !!!
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[0]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS

def VisualizzaCluster(Clusters,quanti):
### NOTA : LA FUNZIONE PRENDE IL NOME DALLA SORGENTE DAI PRIMI CARATTERI DELL'IDENTIFICATIVO DEL RECORD
### specificere in quanti ; esempio se il record è S2_123, quanti =2 per ottenere S2
    vClusters=deepcopy(Clusters)
    vClusters.columns=['ClusterKey', 'ClusterElement']
    vClusters['source']=vClusters['ClusterElement'].astype(str).str[:quanti]

    def Aggregazione(x):
      Campi = {
          '#Sorgenti' :     x['source'].nunique(),
          'Sorgenti' :     x['source'].str.cat(sep=','),
          '#Record' :     x['ClusterElement'].nunique(),
          'Record' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
    groupedCLUSTERS=vClusters.groupby('ClusterKey').apply(Aggregazione).reset_index()

    return groupedCLUSTERS


def _VisualizzaDistribuzioneCluster(Clusters):
    gruppi = Clusters.groupby('ClusterKey')
    conteggio_gruppi = gruppi.size().reset_index(name='NumeroElementiPerCluster')
    Risultato=conteggio_gruppi.groupby('NumeroElementiPerCluster').size().reset_index(name='NumeroCluster')
    print("Numero Elementi", (Risultato['NumeroElementiPerCluster'] * Risultato['NumeroCluster']).sum())
    ClusterMax=conteggio_gruppi[conteggio_gruppi['NumeroElementiPerCluster']==Risultato['NumeroElementiPerCluster'].max()]
    print("Cluster con max numero di elementi:", ClusterMax['ClusterKey'].tolist())

    return Risultato

In [4]:
def CalcolaMatchIndottiCluster(Cluster):
  Join=pd.merge(Cluster,Cluster, on='ClusterKey')
  Join=Join[Join.ClusterElement_x<Join.ClusterElement_y]
  Join=Join[['ClusterElement_x','ClusterElement_y']]
  Join.columns=['l_id','r_id']

  return Join.drop_duplicates()

In [5]:
def stable_marriage(MatchTable:pd.DataFrame):
    MATCH = pd.DataFrame(columns=['l_id', 'r_id', "sim"])
    MT = deepcopy(MatchTable)
    MT = MT.sort_values(["sim"], ascending=[False])
    while True:
        R = MT.loc[(~MT['l_id'].isin(MATCH['l_id'])) & (~MT['r_id'].isin(MATCH['r_id']))]
        if len(R) == 0:
            break
        x = R.iloc[0,:]
        MATCH = MATCH.append(x, ignore_index=True)
    return MATCH

def simmetric_best_match(MatchTable:pd.DataFrame):
  CMT = deepcopy(MatchTable)

  CMT['A_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['l_id']) \
             .cumcount() + 1

  CMT['B_RowNo'] = CMT.sort_values(['sim'], ascending=[False]) \
             .groupby(['r_id']) \
             .cumcount() + 1

  return CMT[(CMT.A_RowNo==1) & (CMT.B_RowNo==1)].drop(columns=['A_RowNo', 'B_RowNo']).sort_values(['sim'], ascending=[False])

In [6]:
def Valuta2(Gold:pd.DataFrame, Match:pd.DataFrame):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta2(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    Gold = Gold[['l_id','r_id']]
    Match = Match[['l_id','r_id']]

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [7]:
def Valuta(Gold:pd.DataFrame, Match:pd.DataFrame):
 #   Gold = Gold[['l_id','r_id']]
 #   Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']
    FOJ = Gold.merge(Match, how='outer', indicator=True)

    TP = FOJ[FOJ['_merge']=='both']
    FP = FOJ[FOJ['_merge']=='right_only']
    FN = FOJ[FOJ['_merge']=='left_only']

    if len(TP) == 0:
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(0,4)],
                'R':[round(0,4)],
                'F':[round(0,4)]
            })
    else:
        P = len(TP)/(len(TP)+len(FP))
        R = len(TP)/(len(TP)+len(FN))
        F = 2 * P * R / ( P + R )
        return pd.DataFrame({
                'MT':[len(Match)],
                'TP':[len(TP)],
                'FP':[len(FP)],
                'FN':[len(FN)],
                'P':[round(P,4)],
                'R':[round(R,4)],
                'F':[round(F,4)]
            })

def VediValuta(Gold:pd.DataFrame, Match:pd.DataFrame, metrics:str):
    #Gold = Gold[['l_id','r_id']]
    #Match = Match[['l_id','r_id']]
    Match = Match.iloc[:, [1, 2]].copy()
    Gold = Gold.iloc[:, [0, 1]].copy()
    Gold.columns=Match.columns=['l_id','r_id']

    FOJ=pd.merge(Gold, Match, how='outer', indicator=True)

    TP=FOJ[FOJ['_merge']=='both']
    FP=FOJ[FOJ['_merge']=='right_only']
    FN=FOJ[FOJ['_merge']=='left_only']

    if metrics == 'FP' :
        return FP
    if metrics == 'TP' :
        return TP
    if metrics == 'FN' :
        return FN

In [8]:
def ValutaBlocking(DA,DB,Block,Gold):
# INPUT : entrambi Block (è il candidate set after blocking)
#        e Gold (Gold Standard) devono essere con due colonne, l_id e r_id
# per avere indipendenza dal nome di queste colonne
# Si suppone che in Block l_id e r_id siano rispettivamente la seconda e la terza colonna
# e che in Gold sia la prima e la seconda
  Block = Block.iloc[:, [1, 2]].copy()
  Gold = Gold.iloc[:, [0, 1]].copy()
  Gold.columns=Block.columns=['l_id','r_id']

  JOIN=pd.merge(Gold, Block)

 # Reduction_Ratio
  RR=1-len(Block)/(len(DA)*len(DB))
 # Pairs Completeness o Recall
  PC = len(JOIN)/len(Gold)
 # Pairs Quality
  PQ = len(JOIN)/len(Block)

  Risultato = pd.DataFrame([(DA.shape[0],DB.shape[0],Block.shape[0],round(RR,4),round(PC,4),round(PQ,4))],
                             columns=['A', 'B', 'BlockSize', 'ReductRatio','PCompletness','PQuality'])

  return Risultato

In [9]:
def IdSOURCES(Sources:list):
  ListaID= []
  for s in Sources.keys():
    ListaID += Sources[s]['id'].to_list()
  return ListaID

# Entity Resolution tra n sorgenti

## 1) ER  tra due sorgenti : BlockingMatchingRule

Si delinea il processo tra due sorgenti nella funzione **BlockingMatchingRule**

In [10]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING

    #...
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING


    return MT

## 2) ER tra tutte le coppie di sorgenti: MatchTableSOURCES

La funzione BlockingMatchingRule viene applicata a tutte le coppie di sorgenti : **MatchTableSOURCES**

`MTSOURCES = MatchTableSOURCES(SOURCES)`

In [11]:
def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping ==> si potrebbe spostare in BlockingMatchingRule
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

## 3) Clusterizzazione: ClusterComponentiConnessi

Si effettua la clusterizzazione dei record sulla base di MTSOURCES usando i componenti connessi
```python
ClusterCalcolati = ClusterComponentiConnessi(
    MTSOURCES[['l_id', 'r_id']],
    IdSOURCES(SOURCES)
)
```
si visualizzano i cluster calcolati

```python
VisualizzaCluster(ClusterCalcolati)
```

e la relativa distribuzione

```python
_VisualizzaDistribuzioneCluster(ClusterCalcolati)
```


## 3) Valutazione tramite **match indotti**

Si confrontano i match indotti dalla clusterizzazione tramite Gold Standard *CalcolaMatchIndottiCluster(ClusterGoldStandard)*
con quelli indotti dalla clusterizzazione calcolata 
*CalcolaMatchIndottiCluster(ClusterCalcolati)*

```python
Valuta(CalcolaMatchIndottiCluster(ClusterGoldStandard),
       CalcolaMatchIndottiCluster(ClusterCalcolati))

```

In [12]:
# Falsi negativi
#VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
# pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

# Falsi Positivi
## VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
## pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')



## Pre-elaborazione : Unione di tutte le sorgenti

Per semplificare il processo, vengono fornite alcune elaborazioni aggiuntive.

Con il seguente codice si effettua l'UNIONE di tutte le sorgneti in SOURCES
si fissano e si verificano le features da usare nel matching 



 Consideriamo la loro unione nel dataframe UNIONE
 il cui schema sarà quello di una sorgente (hanno tutti lo stesso schema)

```python
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)
```



quindi effettuo unione tramite append

```python
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
print(FeaturesList)
```


In [44]:
# per analizzare i MissedMatches  quelli che sono nel GoldStandard e non nel Candidate Set C
#  esplicitamente con il join\n",
#MissedMatches = GoldStandard.merge(C, on=['l_id', 'r_id'], how='left', 
#                                   indicator=True).query("_merge == 'left_only'")[['l_id', 'r_id']]

#pd.merge(pd.merge(MissedMatches,A),B, on='r_id')


# Esempio NCVR2 (clean)

In [12]:
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/NCVR2/'
src_links = [
path+'NCVR_AF_clean.csv',
path+'NCVR_BF_clean.csv',
path+'NCVR_CF_clean.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }
GoldStandardCLEAN=pd.read_csv(path + "GoldStandardCLEAN2.csv")

In [13]:
SOURCES.keys()

dict_keys(['S1', 'S2', 'S3'])

In [14]:
GoldStandardCLEAN

,l_id,r_id
0,0_22_9865350,0_22_9865350
1,0_22_9865350,2_22_3326652
2,2_22_3326652,0_22_9865350
3,2_22_3326652,2_22_3326652
4,0_40_12768214,0_40_12768214
...,...,...
537,2_6340_10171286,2_6340_10171286
538,2_6540_12396430,2_6540_12396430
539,2_6622_13160699,2_6622_13160699
540,2_6631_13273810,2_6631_13273810


In [15]:
# per analizzare il GoldStandard dato , si calcolano i cluster corrispondenti
ClusterGoldStandardCLEAN=ClusterComponentiConnessi(GoldStandardCLEAN[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
VisualizzaCluster(ClusterGoldStandardCLEAN,1).sort_values('#Sorgenti', ascending=False)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
27,28,3,"1,0,2",3,"1_1240_1340473,0_1240_1391987,2_1240_13627651"
3,4,3,"2,1,0",3,"2_140_12924542,1_140_9350108,0_140_9704280"
4,5,3,"0,2,1",3,"0_222_3198122,2_222_12748029,1_222_13226442"
5,6,3,"2,1,0",3,"2_240_13265262,1_240_116265,0_240_94472"
21,22,3,"2,0,1",3,"2_1022_7999709,0_1022_7968679,1_1022_8529932"
...,...,...,...,...,...
23,24,1,0,1,0_1051_2878768
134,135,1,1,1,1_2541_13176126
20,21,1,0,1,0_951_1693517
136,137,1,1,1,1_2640_977594


In [16]:
len(VisualizzaCluster(ClusterGoldStandardCLEAN,1).sort_values('#Sorgenti', ascending=False))
# abbiamo 244 cluster

244

In [17]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandardCLEAN)

Numero Elementi 338
Cluster con max numero di elementi: [4, 5, 6, 19, 22, 23, 27, 28]


,NumeroElementiPerCluster,NumeroCluster
0,1,158
1,2,78
2,3,8


In [18]:
# VisualizzaCluster restituisce un dataframe
df=VisualizzaCluster(ClusterGoldStandardCLEAN,1)
df.columns

Index(['ClusterKey', '#Sorgenti', 'Sorgenti', '#Record', 'Record'], dtype='object')

In [19]:
# che posso analizzare per vedere, ad esempio , se ci sono cluster  in cui il numero di record #Record è > del #Sorgenti
# ovvero se ci sono cluster con più di un elemento della stessa sorgente
df[df['#Sorgenti'] < df['#Record']]

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record


In [45]:
# Consideriamo la loro unione nel dataframe UNIONE
# il cui schema sarà quello di una sorgente (hanno tutti lo stesso schema)
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)
# quindi effettuo unione tramite append
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
FeaturesList

['first_name_first_name_jac_qgm_3_qgm_3',
 'first_name_first_name_cos_dlm_dc0_dlm_dc0',
 'first_name_first_name_jac_dlm_dc0_dlm_dc0',
 'first_name_first_name_mel',
 'first_name_first_name_lev_dist',
 'first_name_first_name_lev_sim',
 'first_name_first_name_nmw',
 'first_name_first_name_sw',
 'last_name_last_name_jac_qgm_3_qgm_3',
 'last_name_last_name_cos_dlm_dc0_dlm_dc0',
 'last_name_last_name_jac_dlm_dc0_dlm_dc0',
 'last_name_last_name_mel',
 'last_name_last_name_lev_dist',
 'last_name_last_name_lev_sim',
 'last_name_last_name_nmw',
 'last_name_last_name_sw',
 'sex_sex_lev_dist',
 'sex_sex_lev_sim',
 'sex_sex_jar',
 'sex_sex_jwn',
 'sex_sex_exm',
 'sex_sex_jac_qgm_3_qgm_3',
 'age_age_lev_dist',
 'age_age_lev_sim',
 'age_age_jar',
 'age_age_jwn',
 'age_age_exm',
 'age_age_jac_qgm_3_qgm_3',
 'birth_place_birth_place_jac_qgm_3_qgm_3',
 'birth_place_birth_place_cos_dlm_dc0_dlm_dc0',
 'birth_place_birth_place_jac_dlm_dc0_dlm_dc0',
 'birth_place_birth_place_mel',
 'birth_place_birth_place_

In [57]:
# Features da considerare nel matching
FixedFeatures = F[F.feature_name.isin(['last_name_last_name_lev_sim',
                                       'zip_code_zip_code_exm',
                                       'first_name_first_name_lev_sim',
                                       'age_age_exm'])]

## Domanda

Partendo dalla regola data nella seguente funzione BlockingMatchinRule, dalla visualizzazione e valutazione dei relativi cluster, fare e commentare opportune modifiche alla funzione BlockingMatchinRule per migliorare precizione e recall. Si possono considerare  solo le FixedFeatures o eventualmente anche le altre disponibili

In [58]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['first_name'] + ' ' + A['last_name']
    B['mix'] = B['first_name'] + ' ' + B['last_name']

    C_SimJoin_mix1  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code'],
                                        r_out_attrs=['first_name', 'last_name', 'sex', 'age', 'birth_place','zip_code']
                                     )
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'l_l_id': 'l_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'r_r_id': 'r_id'})
    C_SimJoin_mix1=C_SimJoin_mix1.rename(columns={'_sim_score': 'sim'})
    cm.set_candset_properties(C_SimJoin_mix1, '_id', 'l_id', 'r_id', A, B)
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule(['last_name_last_name_lev_sim(ltuple, rtuple) >= .7','zip_code_zip_code_exm(ltuple, rtuple) == 1',
                   'age_age_exm(ltuple,rtuple) == 1'], FixedFeatures)
    brm.add_rule([ 'last_name_last_name_lev_sim(ltuple, rtuple) >= .3',
                  'first_name_first_name_lev_sim(ltuple, rtuple) >= .3', 
                  'age_age_exm(ltuple,rtuple) == 1' ], FixedFeatures)
    # in caso di nuova feature aggiungere ad entrambe le regole
    predictions = brm.predict(C_SimJoin_mix1, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

#    MT=C_SimJoin_mix

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [59]:
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

Numero Elementi 338
Cluster con max numero di elementi: [16, 18, 21, 22, 23, 25, 26]


,NumeroElementiPerCluster,NumeroCluster
0,1,179
1,2,69
2,3,7


In [60]:
VisualizzaCluster(ClusterCalcolati,1).sort_values('#Record', ascending=False).head()

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
21,22,3,"2,1,0",3,"2_140_12924542,1_140_9350108,0_140_9704280"
15,16,3,"2,1,0",3,"2_1222_962821,1_1222_1104748,0_1222_1130018"
24,25,3,"0,2,1",3,"0_222_3198122,2_222_12748029,1_222_13226442"
20,21,3,"2,1,0",3,"2_240_13265262,1_240_116265,0_240_94472"
25,26,3,"2,0,1",3,"2_1040_10986386,0_1040_10986345,1_1040_10950068"


In [61]:
# i cluster sono "mediamente più grandi" : nei calcolati ne abbiamo 10 da 3 e 1 da 4
# questo è indice della presenza di falsi negativi
# verifichiamolo con i match indotti
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati))

,MT,TP,FP,FN,P,R,F
0,90,90,0,12,1.0,0.8824,0.9375


In [62]:
# vediamo qualche falso negativo
VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FN')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,0_622_2403315,2_622_6223703,left_only,0_622_2403315,karen,palmer,female,57,ma,28314,2_622_6223703,karen,gollins,female,57,ma,28306
1,0_1140_6460300,2_1140_12513318,left_only,0_1140_6460300,amber,broach,female,32,sc,28115,2_1140_12513318,amber,smith,female,32,sc,27613
2,1_1240_1340473,2_1240_13627651,left_only,1_1240_1340473,teresa,pender,female,63,nc,28570,2_1240_13627651,teresa,starnes,female,63,nc,27562
3,0_1240_1391987,1_1240_1340473,left_only,0_1240_1391987,teresa,starnes,female,63,nc,28570,1_1240_1340473,teresa,pender,female,63,nc,28570
4,0_1740_13855498,2_1740_7342937,left_only,0_1740_13855498,kathren,woodham,female,38,,27530,2_1740_7342937,kathren,hayes,female,38,fl,28209
5,0_3140_907191,2_3140_1207977,left_only,0_3140_907191,angel,johnson,female,41,,28690,2_3140_1207977,angel,mcclellan,female,41,nc,28630
6,0_3822_11913154,1_3822_11891631,left_only,0_3822_11913154,david,troy,male,73,nc,28337,1_3822_11891631,david,troy,male,67,nc,28337
7,0_5640_563518,1_5640_3369336,left_only,0_5640_563518,amanda,conard,female,34,nc,28804,1_5640_3369336,amanda,waldrop,female,34,,27712
8,1_1340_3795261,2_1340_13807668,left_only,1_1340_3795261,denise,hines,female,37,,27801,2_1340_13807668,denise,wells,female,37,nc,27530
9,1_2622_9753093,2_2622_148884,left_only,1_2622_9753093,briana,lefler,female,26,nc,27231,2_2622_148884,briana,orr,female,26,nc,27253


In [39]:
# vediamo qualche falso positivo
VV=VediValuta2(CalcolaMatchIndottiCluster(ClusterGoldStandardCLEAN),CalcolaMatchIndottiCluster(ClusterCalcolati),'FP')
pd.merge(pd.merge(VV,UNIONE, left_on='l_id', right_on='id'),UNIONE, left_on='r_id', right_on='id')

,l_id,r_id,_merge,id_x,first_name_x,last_name_x,sex_x,age_x,birth_place_x,zip_code_x,id_y,first_name_y,last_name_y,sex_y,age_y,birth_place_y,zip_code_y
0,0_3822_11913154,2_2940_9137590,right_only,0_3822_11913154,david,troy,male,73,nc,28337,2_2940_9137590,david,brown,male,51,il,28409
1,1_3822_11891631,2_2940_9137590,right_only,1_3822_11891631,david,troy,male,67,nc,28337,2_2940_9137590,david,brown,male,51,il,28409
2,0_3551_2874784,1_1741_7343057,right_only,0_3551_2874784,christopher,hines,male,21,nc,27360,1_1741_7343057,christopher,miller,male,29,ga,28227
3,0_3751_1179043,1_5722_659298,right_only,0_3751_1179043,james,kelly,male,68,mi,28025,1_5722_659298,james,keefer,male,52,in,28748
4,0_1840_7447052,1_6141_7624860,right_only,0_1840_7447052,joseph,baker,male,46,,28270,1_6141_7624860,joseph,blumberg,male,38,nc,28269
5,1_6141_7624860,2_1840_11606714,right_only,1_6141_7624860,joseph,blumberg,male,38,nc,28269,2_1840_11606714,joseph,baker,male,46,va,28129
6,0_4922_6825581,1_2722_10836165,right_only,0_4922_6825581,christian,banks,male,52,oh,27524,1_2722_10836165,christine,jones,female,36,nc,28347
7,1_2722_10836165,2_4922_11476487,right_only,1_2722_10836165,christine,jones,female,36,nc,28347,2_4922_11476487,christian,banks,male,52,,28393
8,1_1122_9842647,2_2240_934374,right_only,1_1122_9842647,patricia,kurt,female,58,ny,27517,2_2240_934374,latricia,hunter,female,46,nc,28075
9,2_1122_68628,2_2240_934374,right_only,2_1122_68628,patricia,kurt,female,58,ny,27215,2_2240_934374,latricia,hunter,female,46,nc,28075


In [ ]:
# devo aggiungere eta e birth_place 

# Esempio con extract_feature_vecs

In questo esempio il blocking viene fatto senza utilizzare il *similarity* join, quindi dobbiamo calcolare esplicitamente  la similarità delle coppie dei record tramite **extract_feature_vecs**.

Inoltre i questo esempio discuteremo separatamente la parte di blocking e di matching.


In [63]:
# DATASET CLEAN
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/BIKE/'

src_links = [
path+'BikeA.csv',
path+'BikeB.csv']

Attributi = ['id','bike_name', 'km_driven', 'color']
SOURCES = {
    'S' + str(i + 1): pd.read_csv(link).astype(str)[Attributi]
    for i, link in enumerate(src_links)
}


GoldStandard=pd.read_csv(path+ 'BikeGS.csv')

GoldStandard.columns=['l_id','r_id']

In [67]:
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
FeaturesList

['bike_name_bike_name_jac_qgm_3_qgm_3',
 'bike_name_bike_name_cos_dlm_dc0_dlm_dc0',
 'bike_name_bike_name_jac_dlm_dc0_dlm_dc0',
 'bike_name_bike_name_mel',
 'bike_name_bike_name_lev_dist',
 'bike_name_bike_name_lev_sim',
 'bike_name_bike_name_nmw',
 'bike_name_bike_name_sw',
 'km_driven_km_driven_lev_dist',
 'km_driven_km_driven_lev_sim',
 'km_driven_km_driven_jar',
 'km_driven_km_driven_jwn',
 'km_driven_km_driven_exm',
 'km_driven_km_driven_jac_qgm_3_qgm_3',
 'color_color_lev_dist',
 'color_color_lev_sim',
 'color_color_jar',
 'color_color_jwn',
 'color_color_exm',
 'color_color_jac_qgm_3_qgm_3']

In [65]:
ClusterGoldStandard=ClusterComponentiConnessi(GoldStandard[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )

In [66]:
VisualizzaCluster(ClusterGoldStandard,1)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,2,"A,B",2,"A_37934,B_19529"
1,2,2,"A,B",2,"A_40590,B_34527"
2,3,2,"A,B",2,"A_41351,B_10617"
3,4,2,"A,B",2,"A_45737,B_22191"
4,5,2,"B,A",2,"B_20371,A_45319"
...,...,...,...,...,...
223,224,1,B,1,B_17508
224,225,1,A,1,A_41836
225,226,1,A,1,A_37360
226,227,1,A,1,A_42106


In [68]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandard)

Numero Elementi 238
Cluster con max numero di elementi: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


,NumeroElementiPerCluster,NumeroCluster
0,1,218
1,2,10


In [80]:
# blocking & Matching dato
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# AttrEquivalenceBlocker:
    Attributi=['bike_name', 'km_driven', 'color']
    ab = em.AttrEquivalenceBlocker()
    CandidateDato = ab.block_tables(
                          A, B, # dataset
            'color', 'color',
      l_output_attrs=Attributi,
      r_output_attrs=Attributi
    ).rename(columns={'ltable_l_id': 'l_id', 'rtable_r_id': 'r_id'})

    cm.set_candset_properties(CandidateDato, '_id', 'l_id', 'r_id', A, B)

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

    brm = em.BooleanRuleMatcher()
    brm.add_rule([ 'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .4' ], F)
    predictions = brm.predict(CandidateDato, target_attr='pred_label', append=True)
# estrazione del feature_vecs per determinare similarità delle coppie

    MTDato=predictions[predictions.pred_label==1]
    
    cm.set_candset_properties(MTDato, '_id', 'l_id', 'r_id', A, B)
    RuleFeatureTable=F[F['feature_name'].isin(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0'])]
    MTDato=em.extract_feature_vecs(MTDato, feature_table=RuleFeatureTable)
    MTDato = MTDato.rename(columns={'bike_name_bike_name_jac_dlm_dc0_dlm_dc0': 'sim'})

    MTDato=predictions[predictions.pred_label==1]
   

    return MTDato

In [81]:
BlockingMatchingRule(SOURCES['S1'], SOURCES['S2'])

,_id,l_id,r_id,ltable_bike_name,ltable_km_driven,ltable_color,rtable_bike_name,rtable_km_driven,rtable_color,pred_label
131,131,A_37755,B_25914,Honda CB Dazzler Sports,25000,black,Honda CB Unicorn Dazzler Standard,25000,black,1
270,270,A_45737,B_27199,Hero Honda Karizma R,19000,black,Hero Honda Karizma ZMR Standard,17000,black,1
296,296,A_45737,B_23700,Hero Honda Karizma R,19000,black,Hero Honda Karizma ZMR Standard,16000,black,1
325,325,A_44320,B_27491,Hero Honda Splendor Plus,70000,black,Hero Honda Passion Plus Drum,70000,black,1
345,345,A_44320,B_20371,Hero Honda Splendor Plus,70000,black,Hero Honda Passion Plus Drum,40000,black,1
...,...,...,...,...,...,...,...,...,...,...
4855,4855,A_45125,B_8232,Yamaha YZF R15,30000,blue,Yamaha YZF R15 V1,16000,blue,1
4882,4882,A_40346,B_30675,Yamaha Fazer Sport,18000,blue,Yamaha Fazer Standard,26000,blue,1
4889,4889,A_44831,B_27834,Hero Honda Karizma R,16500,yellow,Hero Honda Karizma Standard,10000,yellow,1
4912,4912,A_48730,B_8359,TVS Apache RTR 180 ABS,21000,white,TVS Apache RTR 180 ABS ABS,17000,white,1


In [82]:
# analizziamo e valutiamo la fase di blocking tra due sorgenti SOURCES['S1'],SOURCES['S2']
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# AttrEquivalenceBlocker:
    AttributiOutput=['bike_name',  'km_driven',  'color']
    ab = em.AttrEquivalenceBlocker()
    CandidateDato = ab.block_tables(
            A, B, # dataset
            'color', 'color',
      l_output_attrs=AttributiOutput,
      r_output_attrs=AttributiOutput
    ).rename(columns={'ltable_l_id': 'l_id', 'rtable_r_id': 'r_id'})

    return CandidateDato
C = BlockingMatchingRule(SOURCES['S1'],SOURCES['S2'])
ValutaBlocking(A,B,C,GoldStandard)

,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,238,238,4919,0.9132,0.4,0.0008


In [83]:
# per analizzare i MissedMatches  quelli che sono nel GoldStandard e non nel Candidate Set C
#  esplicitamente con il join\n",
MissedMatches = GoldStandard.merge(C, on=['l_id', 'r_id'], how='left', 
                                   indicator=True).query("_merge == 'left_only'")[['l_id', 'r_id']]

pd.merge(pd.merge(MissedMatches,A),B, on='r_id')

,l_id,r_id,bike_name_x,km_driven_x,color_x,bike_name_y,km_driven_y,color_y
0,A_45737,B_22191,Hero Honda Karizma R,19000,black,Hero Honda Karizma Standard,19000,red
1,A_45319,B_20371,Hero Honda Passion Plus,40000,blue,Hero Honda Passion Plus Drum,40000,black
2,A_47676,B_13781,Hero Honda Glamour,40000,blue,Hero Honda Glamour Drum Self,40000,black
3,A_45125,B_15372,Yamaha YZF R15,30000,blue,Yamaha YZF R15 V1,30000,black
4,A_40827,B_27489,TVS Apache RTR 160,30000,black,TVS Apache RTR 160 Rear Disc Brake,30000,green
5,A_42712,B_7427,Bajaj Discover 100,30000,red,Bajaj Discover 100 5 Speed,30000,black


In [45]:
# tutti quelli persi hanno colore diverso (ovvio dal blocking)
# si nota che hanno lo stesso km_driven, quindi ...

In [84]:
# per migliorare il blocking
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# AttrEquivalenceBlocker:
    AttributiOutput=['bike_name',  'km_driven',  'color']
    ab = em.AttrEquivalenceBlocker()
    CandidateDato = ab.block_tables(
                          A, B, # dataset
            'km_driven', 'km_driven',
      l_output_attrs=AttributiOutput,
      r_output_attrs=AttributiOutput
    ).rename(columns={'ltable_l_id': 'l_id', 'rtable_r_id': 'r_id'})

    return CandidateDato
C = BlockingMatchingRule(SOURCES['S1'],SOURCES['S2'])
ValutaBlocking(A,B,C,GoldStandard)

,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,238,238,284,0.995,1.0,0.0352


In [47]:
# quindi usiamo questo blocking 

In [85]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# AttrEquivalenceBlocker:
    AttributiOutput=['bike_name',  'km_driven',  'color']
    ab = em.AttrEquivalenceBlocker()
    CandidateDato = ab.block_tables(
                          A, B, # dataset
            'km_driven', 'km_driven',
      l_output_attrs=AttributiOutput,
      r_output_attrs=AttributiOutput
    ).rename(columns={'ltable_l_id': 'l_id', 'rtable_r_id': 'r_id'})


    cm.set_candset_properties(C, '_id', 'l_id', 'r_id', A, B)

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
#   print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule([ 'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .4' ], F)
    predictions = brm.predict(C, target_attr='pred_label', append=True)
    MTDato=predictions[predictions.pred_label==1]

    cm.set_candset_properties(MTDato, '_id', 'l_id', 'r_id', A, B)
    RuleFeatureTable=F[F['feature_name'].isin(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0'])]
    MTDato=em.extract_feature_vecs(MTDato, feature_table=RuleFeatureTable)
    MTDato = MTDato.rename(columns={'bike_name_bike_name_jac_dlm_dc0_dlm_dc0': 'sim'})
    print(MTDato.columns)
    return MTDato

In [86]:
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))

Index(['_id', 'l_id', 'r_id', 'sim'], dtype='object')


,MT,TP,FP,FN,P,R,F
0,13,10,3,0,0.7692,1.0,0.8696


In [50]:
# per diminuire FP alzo la soglia

In [87]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# AttrEquivalenceBlocker:
    AttributiOutput=['bike_name',  'km_driven',  'color']
    ab = em.AttrEquivalenceBlocker()
    CandidateDato = ab.block_tables(
                          A, B, # dataset
            'km_driven', 'km_driven',
      l_output_attrs=AttributiOutput,
      r_output_attrs=AttributiOutput
    ).rename(columns={'ltable_l_id': 'l_id', 'rtable_r_id': 'r_id'})


    cm.set_candset_properties(C, '_id', 'l_id', 'r_id', A, B)

    F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
#   print(F['feature_name'].to_list())

    brm = em.BooleanRuleMatcher()
    brm.add_rule([ 'bike_name_bike_name_jac_dlm_dc0_dlm_dc0(ltuple, rtuple) > .5' ], F)
    predictions = brm.predict(C, target_attr='pred_label', append=True)
    MTDato=predictions[predictions.pred_label==1]

    cm.set_candset_properties(MTDato, '_id', 'l_id', 'r_id', A, B)
    RuleFeatureTable=F[F['feature_name'].isin(['bike_name_bike_name_jac_dlm_dc0_dlm_dc0'])]
    MTDato=em.extract_feature_vecs(MTDato, feature_table=RuleFeatureTable)
    MTDato = MTDato.rename(columns={'bike_name_bike_name_jac_dlm_dc0_dlm_dc0': 'sim'})
    print(MTDato.columns)
    return MTDato
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
Valuta2(CalcolaMatchIndottiCluster(ClusterGoldStandard),CalcolaMatchIndottiCluster(ClusterCalcolati))


Index(['_id', 'l_id', 'r_id', 'sim'], dtype='object')


,MT,TP,FP,FN,P,R,F
0,10,10,0,0,1.0,1.0,1.0


In [88]:
MTSOURCES=MatchTableSOURCES(SOURCES)
MTSOURCES

Index(['_id', 'l_id', 'r_id', 'sim'], dtype='object')


,l_id,r_id,sim
0,A_45319,B_20371,0.800000
1,A_37934,B_19529,0.750000
2,A_45125,B_15372,0.750000
3,A_40590,B_34527,0.666667
4,A_41351,B_10617,0.600000
5,A_45737,B_22191,0.600000
6,A_47676,B_13781,0.600000
7,A_44553,B_32225,0.600000
8,A_42712,B_7427,0.600000
9,A_40827,B_27489,0.571429


In [53]:
# a cosa serve la similarità?


# Esempio PPRL (clean)

In [54]:
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/PPRL/'

src_links = [
path+'cleanDatasetA3.csv',
path+'cleanDatasetB3.csv',
path+'cleanDatasetC3.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandard=pd.read_csv(path+ 'cleanGoldStandard3.csv')
GoldStandard.columns=['l_id','r_id']

In [55]:
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
print(FeaturesList)

['given_name_given_name_lev_dist', 'given_name_given_name_lev_sim', 'given_name_given_name_jar', 'given_name_given_name_jwn', 'given_name_given_name_exm', 'given_name_given_name_jac_qgm_3_qgm_3', 'surname_surname_jac_qgm_3_qgm_3', 'surname_surname_cos_dlm_dc0_dlm_dc0', 'surname_surname_jac_dlm_dc0_dlm_dc0', 'surname_surname_mel', 'surname_surname_lev_dist', 'surname_surname_lev_sim', 'surname_surname_nmw', 'surname_surname_sw', 'date_of_birth_date_of_birth_lev_dist', 'date_of_birth_date_of_birth_lev_sim', 'date_of_birth_date_of_birth_jar', 'date_of_birth_date_of_birth_jwn', 'date_of_birth_date_of_birth_exm', 'date_of_birth_date_of_birth_jac_qgm_3_qgm_3']


In [56]:
ClusterGoldStandard=ClusterComponentiConnessi(GoldStandard[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
ClusterGoldStandard.sample()

,ClusterKey,ClusterElement
20,10,S2_3


In [57]:
VisualizzaCluster(ClusterGoldStandard,2)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,2,"S1,S3",2,"S1_0,S3_9"
1,2,2,"S1,S3",2,"S1_2,S3_0"
2,3,2,"S3,S1",2,"S3_1,S1_4"
3,4,3,"S2,S3,S1",3,"S2_0,S3_5,S1_6"
4,5,2,"S1,S3",2,"S1_7,S3_7"
5,6,3,"S3,S1,S2",3,"S3_6,S1_8,S2_4"
6,7,2,"S2,S1",2,"S2_7,S1_10"
7,8,2,"S3,S1",2,"S3_10,S1_11"
8,9,2,"S3,S2",2,"S3_2,S2_5"
9,10,1,S2,1,S2_3


In [58]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandard)

Numero Elementi 26
Cluster con max numero di elementi: [4, 6]


,NumeroElementiPerCluster,NumeroCluster
0,1,6
1,2,7
2,3,2


In [59]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['given_name'] + ' ' + A['surname']
    B['mix'] = B['given_name'] + ' ' + B['surname']

    C  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                        r_out_attrs=['given_name', 'surname', 'date_of_birth']
                                     )
    C = C.rename(columns={'l_l_id': 'l_id', 'r_r_id': 'r_id', '_sim_score': 'sim'})
    cm.set_candset_properties(C, '_id', 'l_id', 'r_id', A, B)
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET C

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule([	'surname_surname_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.3'], F)
    predictions = brm.predict(C, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable

In [60]:
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterCalcolati)


0% [#####] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00
0% [##########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Numero Elementi 26
Cluster con max numero di elementi: [1, 2, 4]


,NumeroElementiPerCluster,NumeroCluster
0,1,3
1,2,7
2,3,3


# Esempio PPRL (dirty)

Confrontare quanto fatto in precedenza nel caso clean con quello che si ottiene nel caso dirty: ripetere le stesse funzioni e fare opportune considerazioni

In [89]:
# DATASET DIRTY
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/PPRL/'

src_links = [
path+'DATASET_small_dirty_A.csv',
path+'DATASET_small_dirty_B.csv',
path+'DATASET_small_dirty_C.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

for x in SOURCES.keys():
          SOURCES[x].columns=['id', 'given_name', 'surname', 'date_of_birth', 'entity']

In [90]:
# il Gold Standard è incluso nei dati; spieghiamo perchè e come costruirlo
SOURCES['S2']

,id,given_name,surname,date_of_birth,entity
0,b_rec-834-dup-0,braecon,schuetz,19440909,rec-834
1,b_rec-1925-dup-3,alexandra,grosvenor,19930305,rec-1925
2,b_rec-1925-dup-0,alexadra,grosvenor,19930305,rec-1925
3,b_rec-1797-org,michael,liersch,19360816,rec-1797
4,b_rec-1925-dup-2,alexandra,grosvenor,19930305,rec-1925
5,b_rec-1654-dup-0,amy,maie r,19040510,rec-1654
6,b_rec-1925-org,alexandra,grosvenor,19930305,rec-1925
7,b_rec-1657-dup-0,olivia,hobson,19760812,rec-1657
8,b_rec-1654-org,amy,clarke,19040510,rec-1654
9,b_rec-1758-org,joshua,green,19010219,rec-1758


In [63]:
# questi dataset sono stati costruiti con il tool GECO https://dmm.anu.edu.au/geco/
# partendo da un record originale rec-1654-org è stato generato un suo duplicato rec-1654-dup-0
# questi due record condividono lo stesso valore di entity rec-1654
# quindi in sostanza entity è quello che si vuole otenere dal  processo di entity resolution
# due record con lo stesso valore di entity sono la stessa persona

In [91]:
UNIONE=pd.DataFrame(columns=SOURCES['S1'].columns)
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])

GoldStandardDIRTY=UNIONE.merge(UNIONE, on='entity')[['id_x','id_y']].query('id_x <= id_y')

GoldStandardDIRTY.columns=['l_id','r_id']
GoldStandardDIRTY.head()

,l_id,r_id
0,a_rec-1799-org,a_rec-1799-org
2,a_rec-1799-org,c_rec-1799-dup-1
3,a_rec-1799-dup-0,a_rec-1799-org
4,a_rec-1799-dup-0,a_rec-1799-dup-0
5,a_rec-1799-dup-0,c_rec-1799-dup-1


In [92]:
ClusterGoldStandardDIRTY=ClusterComponentiConnessi(GoldStandardDIRTY[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
_VisualizzaDistribuzioneCluster(ClusterGoldStandardDIRTY)

Numero Elementi 47
Cluster con max numero di elementi: [3, 5]


,NumeroElementiPerCluster,NumeroCluster
0,1,8
1,2,2
2,3,7
3,4,1
4,5,2


In [93]:
VisualizzaCluster(ClusterGoldStandardDIRTY,1).sort_values("#Record", ascending=False).head()

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
2,3,2,"a,a,c,a,c",5,"a_rec-1668-dup-3,a_rec-1668-dup-2,c_rec-1668-dup-1,a_rec-1668-org,c_rec-1668-dup-0"
4,5,2,"c,c,a,c,a",5,"c_rec-1694-dup-2,c_rec-1694-dup-1,a_rec-1694-dup-0,c_rec-1694-org,a_rec-1694-dup-3"
13,14,1,"b,b,b,b",4,"b_rec-1925-dup-3,b_rec-1925-dup-2,b_rec-1925-dup-0,b_rec-1925-org"
0,1,2,"c,a,a",3,"c_rec-1799-dup-1,a_rec-1799-dup-0,a_rec-1799-org"
16,17,2,"b,c,c",3,"b_rec-1560-org,c_rec-1560-dup-4,c_rec-1560-dup-0"


In [94]:
# 
UNIONE=pd.DataFrame(columns=SOURCES['S2'].columns)
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
print(FeaturesList)

['given_name_given_name_lev_dist', 'given_name_given_name_lev_sim', 'given_name_given_name_jar', 'given_name_given_name_jwn', 'given_name_given_name_exm', 'given_name_given_name_jac_qgm_3_qgm_3', 'surname_surname_jac_qgm_3_qgm_3', 'surname_surname_cos_dlm_dc0_dlm_dc0', 'surname_surname_jac_dlm_dc0_dlm_dc0', 'surname_surname_mel', 'surname_surname_lev_dist', 'surname_surname_lev_sim', 'surname_surname_nmw', 'surname_surname_sw', 'date_of_birth_date_of_birth_lev_dist', 'date_of_birth_date_of_birth_lev_sim', 'date_of_birth_date_of_birth_jar', 'date_of_birth_date_of_birth_jwn', 'date_of_birth_date_of_birth_exm', 'date_of_birth_date_of_birth_jac_qgm_3_qgm_3', 'entity_entity_lev_dist', 'entity_entity_lev_sim', 'entity_entity_jar', 'entity_entity_jwn', 'entity_entity_exm', 'entity_entity_jac_qgm_3_qgm_3']


In [95]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING
    A['mix'] = A['given_name'] + ' ' + A['surname']
    B['mix'] = B['given_name'] + ' ' + B['surname']

    C  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                        r_out_attrs=['given_name', 'surname', 'date_of_birth']
                                     )
    C = C.rename(columns={'l_l_id': 'l_id', 'r_r_id': 'r_id', '_sim_score': 'sim'})
    cm.set_candset_properties(C, '_id', 'l_id', 'r_id', A, B)
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET C

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule([	'surname_surname_jac_qgm_3_qgm_3(ltuple, rtuple) >= 0.3'], F)
    predictions = brm.predict(C, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
MTSOURCES=MatchTableSOURCES(SOURCES)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )

In [96]:
MTSOURCES=MatchTableSOURCES(SOURCES)
MTSOURCES

,l_id,r_id,sim
0,a_rec-1657-dup-4,b_rec-1657-dup-0,1.000000
1,a_rec-1612-org,b_rec-1654-org,1.000000
2,a_rec-456-org,b_rec-1758-org,1.000000
3,a_rec-1797-dup-1,b_rec-1797-org,0.700000
4,a_rec-1225-org,b_rec-1925-dup-3,0.600000
5,a_rec-6140-org,b_rec-6140-dup-0,0.560000
6,a_rec-834-dup-1,b_rec-834-dup-0,0.347826
0,a_rec-8160-org,c_rec-8106-org,1.000000
1,a_rec-1694-dup-0,c_rec-1694-org,1.000000
2,a_rec-456-org,c_rec-1758-dup-1,1.000000


In [97]:
VisualizzaCluster(ClusterCalcolati,1).sort_values("#Record", ascending=False).head()

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,3,"a,c,b",3,"a_rec-1657-dup-4,c_rec-1657-org,b_rec-1657-dup-0"
2,3,3,"b,c,a",3,"b_rec-1758-org,c_rec-1758-dup-1,a_rec-456-org"
1,2,3,"b,a,c",3,"b_rec-1654-org,a_rec-1612-org,c_rec-504-org"
6,7,3,"a,c,b",3,"a_rec-834-dup-1,c_rec-834-org,b_rec-834-dup-0"
9,10,2,"c,a",2,"c_rec-1668-dup-1,a_rec-1668-dup-3"


In [98]:
_VisualizzaDistribuzioneCluster(ClusterCalcolati)

Numero Elementi 47
Cluster con max numero di elementi: [1, 2, 3, 7]


,NumeroElementiPerCluster,NumeroCluster
0,1,13
1,2,11
2,3,4


In [100]:
# perchè questo risultato?

# perchè non ottengo cluster con più di tre elementi?
# con x < y non riesco a trovare i duplicati della stessa source

In [101]:
# esempio simbolico
A = "xxx"
B = "xxxz"
C = "xxxzwk"
D = "zwk"

SOURCES_EM = {
    'S1': pd.DataFrame({'id': ['S1_1'],'X': [A]}),
    'S2': pd.DataFrame({'id': ['S2_1'], 'X': [B]}),
    'S3': pd.DataFrame({'id': ['S3_1','S3_2'], 'X': [C, D]})  
}

In [102]:
# 
UNIONE=pd.DataFrame(columns=SOURCES_EM['S1'].columns)
for x in SOURCES.keys():
          UNIONE=UNIONE.append(SOURCES_EM[x])
A=UNIONE
B=UNIONE
A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })
em.set_key(A, 'l_id')
em.set_key(B, 'r_id')
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
FeaturesList=F['feature_name'].to_list()
print(FeaturesList)


['X_X_lev_dist', 'X_X_lev_sim', 'X_X_jar', 'X_X_jwn', 'X_X_exm', 'X_X_jac_qgm_3_qgm_3']


In [111]:
def BlockingMatchingRule(A,B):
    A=deepcopy(A)
    B=deepcopy(B)
    A=A.rename(columns={'id': "l_id" })
    B=B.rename(columns={'id': "r_id" })

    em.set_key(A, 'l_id')
    em.set_key(B, 'r_id')

# BLOCKING

    C  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                        'X', 'X',  sm.QgramTokenizer(qval=3), threshold=0.2,
                                        l_out_attrs=['X'],
                                        r_out_attrs=['X']
                                     )
    C = C.rename(columns={'l_l_id': 'l_id', 'r_r_id': 'r_id', '_sim_score': 'sim'})
    cm.set_candset_properties(C, '_id', 'l_id', 'r_id', A, B)
############# QUI ABBIAMO OTTENUTO IL CANDIDATE SET C

# MATCHING

    brm = em.BooleanRuleMatcher()
    brm.add_rule([	'X_X_lev_sim(ltuple, rtuple) >= 0.5'], F)
    predictions = brm.predict(C, target_attr='pred_label', append=True)
    MT=predictions[predictions.pred_label==1]

    # estrazione del feature_vecs per determinare similarità delle coppie
    cm.set_candset_properties(MT, '_id', 'l_id', 'r_id', A, B)
    RuleFeatureTable=F[F['feature_name'].isin(['X_X_lev_sim'])]
    MT=em.extract_feature_vecs(MT, feature_table=RuleFeatureTable)
    MT = MT.rename(columns={'X_X_lev_sim': 'sim'})

    
    return MT

def MatchTableSOURCES(Sources:list):
    MatchTable = pd.DataFrame(columns=['l_id', 'r_id', 'sim'])

    for x in Sources.keys():
      for y in Sources.keys():
        if (x<=y): # x<=y nel caso dirty

          MTxy = BlockingMatchingRule(Sources[x], Sources[y])

         # global mapping
          MTxy = stable_marriage(MTxy.query("l_id!=r_id"))
         # MTxy = simmetric_best_match(MTxy.query("l_id!=r_id"))

          MatchTable=MatchTable.append(MTxy[['l_id', 'r_id', 'sim']], sort=True)
    return MatchTable
MTSOURCES=MatchTableSOURCES(SOURCES_EM)
ClusterCalcolati=ClusterComponentiConnessi(MTSOURCES[['l_id','r_id']],
                                                   IdSOURCES(SOURCES_EM) )
MTSOURCES

,l_id,r_id,sim
0,S1_1,S2_1,0.750000
0,S1_1,S3_1,0.500000
0,S2_1,S3_1,0.666667
0,S3_2,S3_1,0.500000
1,S3_1,S3_2,0.500000


In [104]:
ClusterCalcolati

,ClusterKey,ClusterElement
0,1,S3_2
1,1,S1_1
2,1,S2_1
3,1,S3_1


In [105]:

MTs3s3 = BlockingMatchingRule(SOURCES_EM['S3'], SOURCES_EM['S3'])
MTs3s3

,_id,l_id,r_id,sim
0,0,S3_1,S3_1,1.0
1,1,S3_2,S3_1,0.5
2,2,S3_1,S3_2,0.5
3,3,S3_2,S3_2,1.0


In [106]:
stable_marriage(MTs3s3)

,l_id,r_id,sim,_id
0,S3_1,S3_1,1.0,0.0
1,S3_2,S3_2,1.0,3.0


# Esempio Sintetici (clean)

In [85]:
path='http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/Sintetici/'
src_links = [
path+'S1_clean_.csv',
path+'S2_clean_.csv',
path+'S3_clean_.csv']

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

SOURCES['S3']=SOURCES['S3'][SOURCES['S3'].id!='S3_2']
GoldStandardCLEAN=pd.read_csv(path+"GoldStandardClean.csv")
GoldStandardCLEAN.sample()

,l_id,r_id,sim
0,S1_8,S2_4,1.0


In [86]:
SOURCES['S3']

,given_name,surname,date_of_birth,id
0,emmerson,loyck,19211129,S3_0
1,michel,wuchatsch,19190110,S3_1
2,liersch,michael,19360816,S3_3
3,charlotte,hyland,19460401,S3_4
4,braedon,schuetz,19440909,S3_5
5,olivia,hobson,19760812,S3_6
6,joshua,green,19790110,S3_7
7,keely,clarke,19050410,S3_8
8,joshua,morriosn,19101123,S3_9
9,genovefa,hyllande,19071008,S3_11


In [87]:
# IdSOURCES(SOURCES) fornisce tutti i nodi del grafo da usare in ClusterComponentiConnessi
TuttiInodi=IdSOURCES(SOURCES)
print(TuttiInodi)


['S1_0', 'S1_1', 'S1_2', 'S1_3', 'S1_4', 'S1_5', 'S1_6', 'S1_7', 'S1_8', 'S1_9', 'S1_10', 'S1_11', 'S2_0', 'S2_1', 'S2_2', 'S2_3', 'S2_4', 'S2_5', 'S2_6', 'S2_7', 'S3_0', 'S3_1', 'S3_3', 'S3_4', 'S3_5', 'S3_6', 'S3_7', 'S3_8', 'S3_9', 'S3_11']


In [88]:
GoldStandardCLEAN

,l_id,r_id,sim
0,S1_8,S2_4,1.000000
1,S1_9,S2_2,0.793103
2,S1_10,S2_7,0.540541
3,S1_6,S2_0,0.531250
4,S1_3,S2_1,0.526316
5,S1_5,S3_0,1.000000
6,S1_8,S3_6,1.000000
7,S1_4,S3_1,0.862069
8,S1_7,S3_7,0.807692
9,S1_0,S3_9,0.733333


In [89]:
ClusterGoldStandardCLEAN=ClusterComponentiConnessi(GoldStandardCLEAN[['l_id','r_id']],
                                                   IdSOURCES(SOURCES) )
ClusterGoldStandardCLEAN

,ClusterKey,ClusterElement
0,1,S3_6
1,1,S1_8
2,1,S2_4
3,2,S3_3
4,2,S1_9
5,2,S2_2
6,3,S2_7
7,3,S1_10
8,4,S2_0
9,4,S3_5


In [90]:
# per  visualizzare i cluster, si raggruppa sulla ClusterKey
def Aggregazione(x):
      x['source']=x['ClusterElement'].astype(str).str[:2]

      Campi = {
          '#Sources' :     x['source'].nunique(),
          'Sources' :     x['source'].drop_duplicates().str.cat(sep=','),
          '#Elements' :     x['ClusterElement'].nunique(),
          'Elements' :     x['ClusterElement'].str.cat(sep=',')
          }
      return pd.Series(Campi)
ClusterGoldStandardCLEAN.groupby('ClusterKey').apply(Aggregazione).reset_index().sort_values('#Sources', ascending=False)

,ClusterKey,#Sources,Sources,#Elements,Elements
0,1,3,"S3,S1,S2",3,"S3_6,S1_8,S2_4"
1,2,3,"S3,S1,S2",3,"S3_3,S1_9,S2_2"
3,4,3,"S2,S3,S1",3,"S2_0,S3_5,S1_6"
7,8,3,"S1,S3,S2",3,"S1_7,S3_7,S2_5"
2,3,2,"S2,S1",2,"S2_7,S1_10"
4,5,2,"S1,S2",2,"S1_3,S2_1"
5,6,2,"S1,S3",2,"S1_5,S3_0"
6,7,2,"S3,S1",2,"S3_1,S1_4"
8,9,2,"S1,S3",2,"S1_0,S3_9"
9,10,2,"S1,S3",2,"S1_11,S3_11"


In [94]:
VisualizzaCluster(ClusterGoldStandardCLEAN,2)

,ClusterKey,#Sorgenti,Sorgenti,#Record,Record
0,1,3,"S3,S1,S2",3,"S3_6,S1_8,S2_4"
1,2,3,"S3,S1,S2",3,"S3_3,S1_9,S2_2"
2,3,2,"S2,S1",2,"S2_7,S1_10"
3,4,3,"S2,S3,S1",3,"S2_0,S3_5,S1_6"
4,5,2,"S1,S2",2,"S1_3,S2_1"
5,6,2,"S1,S3",2,"S1_5,S3_0"
6,7,2,"S3,S1",2,"S3_1,S1_4"
7,8,3,"S1,S3,S2",3,"S1_7,S3_7,S2_5"
8,9,2,"S1,S3",2,"S1_0,S3_9"
9,10,2,"S1,S3",2,"S1_11,S3_11"


In [95]:
_VisualizzaDistribuzioneCluster(ClusterGoldStandardCLEAN)

Numero Elementi 30
Cluster con max numero di elementi: [1, 2, 4, 8]


,NumeroElementiPerCluster,NumeroCluster
0,1,4
1,2,7
2,3,4


In [96]:
# GoldStandardCLEAN è riferito a tutte e tre le sorgenti, e contiene solo le coppie  Si_, Sj, con i < j
GoldStandardCLEAN.sample(5)

,l_id,r_id,sim
3,S1_6,S2_0,0.531250
2,S1_10,S2_7,0.540541
5,S1_5,S3_0,1.000000
11,S1_6,S3_5,0.531250
9,S1_0,S3_9,0.733333


In [97]:
# per ottenere il gold standard relativo ad   una coppia di sorgenti
def GoldStandard(GS,s1,s2):
    return GS[ (GS['l_id'].isin(SOURCES[s1]['id'])) & GS['r_id'].isin(SOURCES[s2]['id'])]

In [98]:
GoldStandard(GoldStandardCLEAN,'S1','S3')

,l_id,r_id,sim
5,S1_5,S3_0,1.000000
6,S1_8,S3_6,1.000000
7,S1_4,S3_1,0.862069
8,S1_7,S3_7,0.807692
9,S1_0,S3_9,0.733333
10,S1_11,S3_11,0.676471
11,S1_6,S3_5,0.531250
12,S1_9,S3_3,0.529412


## Matching tra una coppia di sources

In [99]:
# si considera la coppia di sources
A=SOURCES['S1']
B=SOURCES['S3']

A=A.rename(columns={'id': "l_id" })
B=B.rename(columns={'id': "r_id" })

em.set_key(A, 'l_id')
em.set_key(B, 'r_id')

True

In [100]:
# BLOCKING

A['mix'] = A['given_name'] + ' ' + A['surname'] + ' ' + A['date_of_birth']
B['mix'] = B['given_name'] + ' ' + B['surname'] + ' ' + B['date_of_birth']

C_SimJoin_mix  = ssj.jaccard_join(A, B, 'l_id', 'r_id',
                                    'mix', 'mix',  sm.QgramTokenizer(qval=3), threshold=0.3,
                                    l_out_attrs=['given_name', 'surname', 'date_of_birth'],
                                    r_out_attrs=['given_name', 'surname', 'date_of_birth'])
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'l_l_id': 'l_id'})
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'r_r_id': 'r_id'})
C_SimJoin_mix=C_SimJoin_mix.rename(columns={'_sim_score': 'sim'})

em.set_key(C_SimJoin_mix, '_id')
em.set_ltable(C_SimJoin_mix, A)
em.set_rtable(C_SimJoin_mix, B)
em.set_fk_ltable(C_SimJoin_mix, 'l_id')
em.set_fk_rtable(C_SimJoin_mix, 'r_id')
C_SimJoin_mix

0% [##########] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069
3,3,S1_9,S3_3,michael,lierach,19360816,liersch,michael,19360816,0.529412
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471


In [101]:
C_SimJoin_mix.columns
GS=GoldStandard(GoldStandardCLEAN,'S1','S3')
GS.columns

Index(['l_id', 'r_id', 'sim'], dtype='object')

In [102]:
ValutaBlocking(A,B,C_SimJoin_mix,GoldStandard(GoldStandardCLEAN,'S1','S3'))

,A,B,BlockSize,ReductRatio,PCompletness,PQuality
0,12,10,9,0.925,1.0,0.8889


In [103]:
# Matching

In [104]:
# Codice completo e generale per ottenere le  feature_table
block_c = em.get_attr_corres(A, B)
atypesA = em.get_attr_types(A)
atypesB = em.get_attr_types(B)
for c in A.columns:
    atypesB[c]=atypesA[c]

feature_table = em.get_features(A, B,
                                 atypesA, atypesB,
                                 block_c,
                                 em.get_tokenizers_for_matching(),
                                 em.get_sim_funs_for_matching())
print(feature_table['feature_name'].to_list())

['given_name_given_name_lev_dist', 'given_name_given_name_lev_sim', 'given_name_given_name_jar', 'given_name_given_name_jwn', 'given_name_given_name_exm', 'given_name_given_name_jac_qgm_3_qgm_3', 'surname_surname_lev_dist', 'surname_surname_lev_sim', 'surname_surname_jar', 'surname_surname_jwn', 'surname_surname_exm', 'surname_surname_jac_qgm_3_qgm_3', 'date_of_birth_date_of_birth_lev_dist', 'date_of_birth_date_of_birth_lev_sim', 'date_of_birth_date_of_birth_jar', 'date_of_birth_date_of_birth_jwn', 'date_of_birth_date_of_birth_exm', 'date_of_birth_date_of_birth_jac_qgm_3_qgm_3', 'mix_mix_jac_qgm_3_qgm_3', 'mix_mix_cos_dlm_dc0_dlm_dc0', 'mix_mix_jac_dlm_dc0_dlm_dc0', 'mix_mix_mel', 'mix_mix_lev_dist', 'mix_mix_lev_sim', 'mix_mix_nmw', 'mix_mix_sw']


In [105]:
# in questi esempi simbolici usiamo direttamente
feature_table = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)

In [106]:
# DOMANDA: commentare il significato della seguente rule e valutarne il risultato

brm = em.BooleanRuleMatcher()
brm.add_rule(['date_of_birth_date_of_birth_lev_dist(ltuple, rtuple) == 1',
              ], feature_table)


brm.add_rule(['surname_surname_lev_sim(ltuple, rtuple)*0.5 \
                + given_name_given_name_lev_sim(ltuple, rtuple)*0.2 \
                 + date_of_birth_date_of_birth_jac_qgm_3_qgm_3(ltuple, rtuple)*0.3 > 0.7'], feature_table)
predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
MT

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815,1
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000,1
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069,1
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250,1
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000,1
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692,1
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333,1
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471,1


In [107]:
Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)

,MT,TP,FP,FN,P,R,F
0,8,7,1,1,0.875,0.875,0.875


In [108]:
#Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3')[['l_id','r_id']],MT[['l_id','r_id']])

In [109]:
VV=VediValuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FP')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,given_name_y,surname_y,date_of_birth_y,mix_y
0,S1_2,S3_0,right_only,emmerson,lock,19211129,emmerson lock 19211129,emmerson,loyck,19211129,emmerson loyck 19211129


In [110]:
VV=VediValuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FP')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,given_name_y,surname_y,date_of_birth_y,mix_y
0,S1_2,S3_0,right_only,emmerson,lock,19211129,emmerson lock 19211129,emmerson,loyck,19211129,emmerson loyck 19211129


In [111]:
VV=VediValuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FN')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,given_name_y,surname_y,date_of_birth_y,mix_y
0,S1_9,S3_3,left_only,michael,lierach,19360816,michael lierach 19360816,liersch,michael,19360816,liersch michael 19360816


In [112]:
# come considerare questo falso negativo dovuto allo scambio tra nome e cognome?
# definire NAME come concatenazione ed usare una jaccard
# ()

In [113]:
# non è necessario rifare blocking, ma solo definire il nuovo attributo
A['NAME'] = A['given_name'] + ' ' + A['surname']
B['NAME'] = B['given_name'] + ' ' + B['surname']

# e rigenerare le features, verificando le features per il nuovo attributo
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)


In [114]:
F = em.get_features_for_matching(A, B, validate_inferred_attr_types=False)
brm = em.BooleanRuleMatcher()
brm.add_rule(['surname_surname_exm(ltuple, rtuple) == 1', 'date_of_birth_date_of_birth_lev_dist(ltuple, rtuple) == 1'], F)
brm.add_rule(['NAME_NAME_jac_qgm_3_qgm_3(ltuple, rtuple)*0.6 \
                 + date_of_birth_date_of_birth_jac_qgm_3_qgm_3(ltuple, rtuple)*0.4 > 0.5'], F)
predictions = brm.predict(C_SimJoin_mix, target_attr='pred_label', append=True)
MT=predictions[predictions.pred_label==1]
MT

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815,1
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000,1
2,2,S1_4,S3_1,michael,wuchatsch,19190110,michel,wuchatsch,19190110,0.862069,1
3,3,S1_9,S3_3,michael,lierach,19360816,liersch,michael,19360816,0.529412,1
4,4,S1_6,S3_5,rhys,schuetz,19440909,braedon,schuetz,19440909,0.531250,1
5,5,S1_8,S3_6,olivia,hobson,19760812,olivia,hobson,19760812,1.000000,1
6,6,S1_7,S3_7,joshua,greenj,19790110,joshua,green,19790110,0.807692,1
7,7,S1_0,S3_9,joshua,morrison,19101123,joshua,morriosn,19101123,0.733333,1
8,8,S1_11,S3_11,genoveffa,hylander,19071008,genovefa,hyllande,19071008,0.676471,1


In [115]:
Valuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)

,MT,TP,FP,FN,P,R,F
0,9,8,1,0,0.8889,1.0,0.9412


In [116]:
VV=VediValuta2(GoldStandard(GoldStandardCLEAN,'S1','S3'),
              MT,'FP')
pd.merge(pd.merge(VV,A, on='l_id'),B, on='r_id').head()

,l_id,r_id,_merge,given_name_x,surname_x,date_of_birth_x,mix_x,NAME_x,given_name_y,surname_y,date_of_birth_y,mix_y,NAME_y
0,S1_2,S3_0,right_only,emmerson,lock,19211129,emmerson lock 19211129,emmerson lock,emmerson,loyck,19211129,emmerson loyck 19211129,emmerson loyck


In [117]:
# 

In [118]:
# 

In [119]:
GS=GoldStandard(GoldStandardCLEAN,'S1','S3')
# Uno-a-molti (l_id → più r_id)
GS[GS.duplicated('l_id', keep=False)]

,l_id,r_id,sim


In [120]:
# Molti-a-uno (r_id → più l_id)
GS[GS.duplicated('r_id', keep=False)]

,l_id,r_id,sim


In [121]:
MT[MT.duplicated('r_id', keep=False)]

,_id,l_id,r_id,l_given_name,l_surname,l_date_of_birth,r_given_name,r_surname,r_date_of_birth,sim,pred_label
0,0,S1_2,S3_0,emmerson,lock,19211129,emmerson,loyck,19211129,0.814815,1
1,1,S1_5,S3_0,emmerson,loyck,19211129,emmerson,loyck,19211129,1.000000,1


In [ ]:
MT[MT.duplicated('l_id', keep=False)]

In [ ]:
MT=stable_marriage(MT)

In [ ]:
MT
# Porta 'id' come prima colonna
cols = ['_id'] + [col for col in MT.columns if col != '_id']
MT = MT[cols]
MT


In [ ]:
Valuta(GoldStandard(GoldStandardCLEAN,'S1','S3'),MT)